# What PyTorch tensors add to NumPy arrays

A parking garage exports a week of occupancy counts: 4 levels, 24 hours, 7 days, 120 spaces per level. You turn that NumPy array into a PyTorch tensor, predict the dtype of mixed operations, move it to a device, find out which views work and which need a copy, and fix three real errors.

About 10 minutes. It runs on a free Colab CPU runtime; no graphics chip, account or key needed.

## 1. The week of counts

The next cell makes the same data as the lesson: random whole numbers from 0 to 120, seeded so you get the same numbers. It also defines `show`, a small helper that prints the dtype, shape and device of a tensor, or the error message when a call fails.

Look for: the NumPy array is `int64` with shape `(4, 24, 7)`. The lesson used torch 2.15.0; your version may be different, and a few messages may be worded a little differently.

In [1]:
import numpy as np
import torch

rng = np.random.default_rng(0)
LEVELS, HOURS, DAYS, CAPACITY = 4, 24, 7, 120
# a week of occupancy counts, as the garage's export gives them: whole numbers, NumPy's default int64
week_counts = rng.integers(0, CAPACITY + 1, size=(LEVELS, HOURS, DAYS))

def show(label, fn):
    try:
        r = fn()
        if isinstance(r, torch.Tensor):
            print(f"{label} -> dtype={r.dtype} shape={tuple(r.shape)} device={r.device}")
        else:
            print(f"{label} -> {r!r}")
    except Exception as e:
        print(f"{label} -> {type(e).__name__}: {e}")

print("torch version:", torch.__version__)
show("week_counts (numpy)", lambda: (str(week_counts.dtype), week_counts.shape))

torch version: 2.15.0
week_counts (numpy) -> ('int64', (4, 24, 7))


## 2. What the tensor carries

`torch.from_numpy` turns the array into a tensor. The cell then compares the default number types of the two libraries.

Look for: `counts` keeps NumPy's `int64`, strides `(168, 7, 1)`, contiguous `True`. PyTorch's default float is `float32` (4 bytes per number), NumPy's is `float64` (8 bytes). The NumPy division gives `float64`.

In [2]:
counts = torch.from_numpy(week_counts)
show("counts = torch.from_numpy(week_counts)", lambda: counts)
show("counts.stride()", lambda: counts.stride())
show("counts.is_contiguous()", lambda: counts.is_contiguous())
show("counts.element_size()", lambda: counts.element_size())
show("torch.get_default_dtype()", lambda: torch.get_default_dtype())
show("counts.float().element_size()", lambda: counts.float().element_size())
show("torch.ones(3, dtype=torch.float64).element_size()", lambda: torch.ones(3, dtype=torch.float64).element_size())
show("np.ones(3).dtype", lambda: str(np.ones(3).dtype))
show("torch.ones(3).dtype", lambda: torch.ones(3).dtype)
show("week_counts / 2 (numpy)", lambda: str((week_counts / 2).dtype))

counts = torch.from_numpy(week_counts) -> dtype=torch.int64 shape=(4, 24, 7) device=cpu
counts.stride() -> (168, 7, 1)
counts.is_contiguous() -> True
counts.element_size() -> 8
torch.get_default_dtype() -> torch.float32
counts.float().element_size() -> 4
torch.ones(3, dtype=torch.float64).element_size() -> 8
np.ones(3).dtype -> 'float64'
torch.ones(3).dtype -> torch.float32
week_counts / 2 (numpy) -> 'float64'


## 3. Six mixed operations

These are the six operations from the first screen of the lesson. `rates` is a NumPy `float64` array of the same shape.

Look for: `counts / 2` is `float32` (NumPy gave `float64`), `counts // 2` stays `int64`, adding a float64 tensor with dimensions gives `float64`, adding a single float64 number to a float32 tensor stays `float32`. `counts.mean()` fails with a message that ends in `Got: Long`, and `counts.float().mean()` works.

In [3]:
rates = np.full((LEVELS, HOURS, DAYS), 1.0)  # numpy float64
show("1 counts / 2", lambda: counts / 2)
show("2 counts // 2", lambda: counts // 2)
show("3 counts * 0.5", lambda: counts * 0.5)
show("4 counts + torch.from_numpy(rates)", lambda: counts + torch.from_numpy(rates))
show("5 counts.float() + torch.tensor(1.0, dtype=torch.float64)", lambda: counts.float() + torch.tensor(1.0, dtype=torch.float64))
show("6 counts.mean()", lambda: counts.mean())
show("6b counts.float().mean()", lambda: counts.float().mean())
show("torch.result_type(counts, 0.5)", lambda: torch.result_type(counts, 0.5))

1 counts / 2 -> dtype=torch.float32 shape=(4, 24, 7) device=cpu
2 counts // 2 -> dtype=torch.int64 shape=(4, 24, 7) device=cpu
3 counts * 0.5 -> dtype=torch.float32 shape=(4, 24, 7) device=cpu
4 counts + torch.from_numpy(rates) -> dtype=torch.float64 shape=(4, 24, 7) device=cpu
5 counts.float() + torch.tensor(1.0, dtype=torch.float64) -> dtype=torch.float32 shape=(4, 24, 7) device=cpu
6 counts.mean() -> NotImplementedError: mean(): could not infer output dtype. Input dtype must be either a floating point or complex dtype. Got: Long
6b counts.float().mean() -> dtype=torch.float32 shape=() device=cpu
torch.result_type(counts, 0.5) -> torch.float32


## 4. Devices: `.to()` returns a new tensor

The lesson ran on a Mac, where the graphics chip is called `mps`. Colab's free runtime has no graphics chip, so the next cell picks `cuda` if there is one and `cpu` otherwise. Then it moves a tensor without keeping the result, and with keeping it.

Look for: after the bare `x.to(device)` line, `x.device` still prints `cpu`. On a CPU runtime the "move" changes nothing: `x_gpu` is `x` itself, so the memory check prints `True`. On a graphics chip it prints `False`, because moving copies every number, as in the lesson. `x.to('cpu') is x` prints `True` either way.

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

x = counts.float()
show("x.device", lambda: x.device)
x.to(device)
show("after `x.to(device)` (result not kept): x.device", lambda: x.device)
x_gpu = x.to(device)
show("x_gpu = x.to(device); x_gpu.device", lambda: x_gpu.device)
show("x_gpu.data_ptr() == x.data_ptr()", lambda: x_gpu.data_ptr() == x.data_ptr())
show("x.to('cpu') is x", lambda: x.to("cpu") is x)

device: cpu
x.device -> device(type='cpu')
after `x.to(device)` (result not kept): x.device -> device(type='cpu')
x_gpu = x.to(device); x_gpu.device -> device(type='cpu')
x_gpu.data_ptr() == x.data_ptr() -> True
x.to('cpu') is x -> True


## 5. Transpose moves nothing

`counts.transpose(1, 2)` gives each level as days x hours. It copies no numbers; only the strides change.

Look for: shape `(4, 7, 24)`, strides `(168, 1, 7)` (the day and hour steps swapped), contiguous `False`, and the same memory address as `counts` (`True`).

In [5]:
by_day = counts.transpose(1, 2)  # levels x days x hours
show("by_day = counts.transpose(1, 2)", lambda: by_day)
show("by_day.stride()", lambda: by_day.stride())
show("by_day.is_contiguous()", lambda: by_day.is_contiguous())
show("by_day.data_ptr() == counts.data_ptr()", lambda: by_day.data_ptr() == counts.data_ptr())

by_day = counts.transpose(1, 2) -> dtype=torch.int64 shape=(4, 7, 24) device=cpu
by_day.stride() -> (168, 1, 7)
by_day.is_contiguous() -> False
by_day.data_ptr() == counts.data_ptr() -> True


## 6. The four view calls

`.view(shape)` gives the same memory a new shape without copying. It works only if new strides can describe it. Splitting one dimension always works; merging two neighbours works only when their steps line up.

Look for: V1 and V3 fail with "view size is not compatible with input tensor's size and stride", because they merge days and hours. V2 and V4 work, because they only split one dimension. `reshape(4, 168)` works, and so does `.contiguous().view(4, 168)`.

In [6]:
show("V1 by_day.view(4, 168)", lambda: by_day.view(4, 168))
show("V2 by_day.view(2, 2, 7, 24)", lambda: by_day.view(2, 2, 7, 24))
show("V3 by_day.view(-1)", lambda: by_day.view(-1))
show("V4 by_day.view(4, 7, 4, 6)", lambda: by_day.view(4, 7, 4, 6))
show("by_day.view(2, 2, 7, 24).stride()", lambda: by_day.view(2, 2, 7, 24).stride())
show("by_day.view(4, 7, 4, 6).stride()", lambda: by_day.view(4, 7, 4, 6).stride())
show("by_day.reshape(4, 168)", lambda: by_day.reshape(4, 168))
show("by_day.contiguous().view(4, 168)", lambda: by_day.contiguous().view(4, 168))

V1 by_day.view(4, 168) -> RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.
V2 by_day.view(2, 2, 7, 24) -> dtype=torch.int64 shape=(2, 2, 7, 24) device=cpu
V3 by_day.view(-1) -> RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.
V4 by_day.view(4, 7, 4, 6) -> dtype=torch.int64 shape=(4, 7, 4, 6) device=cpu
by_day.view(2, 2, 7, 24).stride() -> (336, 168, 1, 7)
by_day.view(4, 7, 4, 6).stride() -> (168, 1, 42, 7)
by_day.reshape(4, 168) -> dtype=torch.int64 shape=(4, 168) device=cpu
by_day.contiguous().view(4, 168) -> dtype=torch.int64 shape=(4, 168) device=cpu


## 7. reshape: a view or a copy?

`reshape` returns a view when the strides allow it and a copy when they do not. The cell finds out which one it got by writing into the result. It works on a fresh copy of the counts, `c2`.

Look for: `flat_a` shares memory with `c2`, so writing 999 into it changes `c2[0,0,0]` from 102 to 999. `flat_b` is a copy, so writing 555 into it leaves `c2[0,1,0]` at 1. The last line shows that `torch.from_numpy` shares memory too: changing the NumPy array changed the tensor to 7.

In [7]:
c2 = torch.from_numpy(week_counts.copy())
flat_a = c2.reshape(4, 168)                  # c2 is contiguous
flat_b = c2.transpose(1, 2).reshape(4, 168)  # needs a copy
show("flat_a shares storage with c2", lambda: flat_a.data_ptr() == c2.data_ptr())
show("flat_b shares storage with c2", lambda: flat_b.data_ptr() == c2.data_ptr())
before = int(c2[0, 0, 0])
flat_a[0, 0] = 999
show("c2[0,0,0] before", lambda: before)
show("c2[0,0,0] after flat_a[0,0] = 999", lambda: int(c2[0, 0, 0]))
before_b = int(c2[0, 1, 0])
flat_b[0, 1] = 555
show("c2[0,1,0] before flat_b write", lambda: before_b)
show("c2[0,1,0] after flat_b[0,1] = 555 (flat_b[0,1] is day 0 hour 1)", lambda: int(c2[0, 1, 0]))
arr = week_counts.copy(); t = torch.from_numpy(arr); arr[0, 0, 0] = 7
show("torch.from_numpy then numpy write: t[0,0,0]", lambda: int(t[0, 0, 0]))

flat_a shares storage with c2 -> True
flat_b shares storage with c2 -> False
c2[0,0,0] before -> 102
c2[0,0,0] after flat_a[0,0] = 999 -> 999
c2[0,1,0] before flat_b write -> 1
c2[0,1,0] after flat_b[0,1] = 555 (flat_b[0,1] is day 0 hour 1) -> 1
torch.from_numpy then numpy write: t[0,0,0] -> 7


## 8. The preparation as one function, line by line

This is `to_model_input` from the lesson's code screen. The helper `trace_run` runs it and, before each line, records the five facts of each tensor: dtype, shape, strides, contiguous, device. You do not need to read the helper; read the printed steps.

Look for: `permute` makes `by_day` with strides `(1, 168, 7)`, not contiguous. `reshape(DAYS, -1)` gives shape `(7, 96)` with strides `(1, 7)`: still a view. `.float()` changes the dtype to `float32`. `.to(device)` changes the device (here it stays `cpu`; on the lesson's Mac it became `mps:0`). The last three lines check the reshape step on its own: it shares memory with `counts` (`True`), so it was a view, not a copy.

In [8]:
import sys

def summary(v):
    return {"dtype": str(v.dtype).replace("torch.", ""), "shape": tuple(v.shape), "stride": v.stride(),
            "contiguous": v.is_contiguous(), "device": str(v.device)}

def trace_run(fn, *args, keep=None):
    steps = []
    first = fn.__code__.co_firstlineno
    def local(frame, event, arg):
        if event in ("line", "return"):
            vs = {k: summary(v) for k, v in frame.f_locals.items() if keep is None or k in keep}
            steps.append((frame.f_lineno - first, event, vs))
        return local
    sys.settrace(lambda frame, event, arg: local if frame.f_code is fn.__code__ else None)
    try:
        result = fn(*args)
    finally:
        sys.settrace(None)
    return result, steps

def to_model_input(week_counts, device):
    counts = torch.from_numpy(week_counts)
    by_day = counts.permute(2, 0, 1)
    rows = by_day.reshape(DAYS, -1)
    rows = rows.float() / CAPACITY
    rows = rows.to(device)
    return rows

result, steps = trace_run(to_model_input, week_counts, device, keep=["counts", "by_day", "rows"])
code_lines = {1: "counts = torch.from_numpy(week_counts)", 2: "by_day = counts.permute(2, 0, 1)",
              3: "rows = by_day.reshape(DAYS, -1)", 4: "rows = rows.float() / CAPACITY", 5: "rows = rows.to(device)"}
for line, event, vs in steps[1:]:
    if event == "line":   # about to run `line`, so line - 1 has just run
        ran = code_lines[line - 1]
        name = ran.split(" = ")[0]
        print(f"after `{ran}`")
        print("   ", name, vs[name])

# the reshape step on its own: does it share memory with counts?
counts = torch.from_numpy(week_counts)
rows = counts.permute(2, 0, 1).reshape(DAYS, -1)
show("counts.permute(2, 0, 1).reshape(7, -1)", lambda: rows)
show("its stride()", lambda: rows.stride())
show("shares storage with counts", lambda: rows.data_ptr() == counts.data_ptr())

after `counts = torch.from_numpy(week_counts)`
    counts {'dtype': 'int64', 'shape': (4, 24, 7), 'stride': (168, 7, 1), 'contiguous': True, 'device': 'cpu'}
after `by_day = counts.permute(2, 0, 1)`
    by_day {'dtype': 'int64', 'shape': (7, 4, 24), 'stride': (1, 168, 7), 'contiguous': False, 'device': 'cpu'}
after `rows = by_day.reshape(DAYS, -1)`
    rows {'dtype': 'int64', 'shape': (7, 96), 'stride': (1, 7), 'contiguous': False, 'device': 'cpu'}
after `rows = rows.float() / CAPACITY`
    rows {'dtype': 'float32', 'shape': (7, 96), 'stride': (1, 7), 'contiguous': False, 'device': 'cpu'}
after `rows = rows.to(device)`
    rows {'dtype': 'float32', 'shape': (7, 96), 'stride': (1, 7), 'contiguous': False, 'device': 'cpu'}
counts.permute(2, 0, 1).reshape(7, -1) -> dtype=torch.int64 shape=(7, 96) device=cpu
its stride() -> (1, 7)
shares storage with counts -> True


## 9. Feed the model

A layer `nn.Linear(96, 1)` on the same device takes the prepared rows.

Look for: output shape `(7, 1)`, one number per day.

In [9]:
torch.manual_seed(0)
model96 = torch.nn.Linear(LEVELS * HOURS, 1).to(device)
print("model(result).shape:", tuple(model96(result).shape))

model(result).shape: (7, 1)


## 10. Three real errors, broken on purpose

Each line fails on purpose, and `show` prints the message instead of stopping the notebook. Then the fix runs.

Look for:
- Error 1 names two dtypes: `Double` (float64, from NumPy) and `Float` (float32, the weights). `.float()` fixes it.
- Error 2 is a device mismatch. With no graphics chip, the cell puts the model on PyTorch's `meta` device (a pretend device with no real memory) so you still see one. The wording differs from the lesson's Mac message, but it also names the two devices. Moving the input to the model's device fixes it (skipped on `meta`, which has no numbers to compute with).
- Error 3 is the view error after a transpose. `reshape` fixes it.

In [10]:
torch.manual_seed(0)
model = torch.nn.Linear(HOURS, 1)  # predicts from 24 hourly values
show("model.weight.dtype", lambda: model.weight.dtype)
day_np = (week_counts[0, :, 0] / CAPACITY).reshape(1, HOURS)  # numpy float64
show("E1 model(torch.from_numpy(day_np))", lambda: model(torch.from_numpy(day_np)))
show("E1 fixed model(torch.from_numpy(day_np).float())", lambda: model(torch.from_numpy(day_np).float()))

other = device if device != "cpu" else "meta"
model_gpu = torch.nn.Linear(HOURS, 1).to(other)
day = torch.from_numpy(day_np).float()
show(f"E2 model on {other}, model_gpu(day)", lambda: model_gpu(day))
if other != "meta":
    show(f"E2 fixed model_gpu(day.to('{other}'))", lambda: model_gpu(day.to(other)))

show("E3 counts.transpose(1, 2).view(LEVELS, -1)", lambda: counts.transpose(1, 2).view(LEVELS, -1))
show("E3 fixed counts.transpose(1, 2).reshape(LEVELS, -1)", lambda: counts.transpose(1, 2).reshape(LEVELS, -1))

model.weight.dtype -> torch.float32
E1 model(torch.from_numpy(day_np)) -> RuntimeError: mat1 and mat2 must have the same dtype, but got Double and Float
E1 fixed model(torch.from_numpy(day_np).float()) -> dtype=torch.float32 shape=(1, 1) device=cpu


E2 model on meta, model_gpu(day) -> RuntimeError: Tensor on device meta is not on the expected device cpu!
E3 counts.transpose(1, 2).view(LEVELS, -1) -> RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.
E3 fixed counts.transpose(1, 2).reshape(LEVELS, -1) -> dtype=torch.int64 shape=(4, 168) device=cpu


## Now change it

Three small experiments. Each cell has one line marked `# <- edit`. Change it, run the cell, and compare.

**Experiment 1: what kind of number is the scale?** The cell multiplies the int64 counts by `scale`. It prints `float32` now. Change `scale` to `2` and the result stays `int64`: a whole number times whole numbers needs no fractions. Change it to `torch.tensor(0.5, dtype=torch.float64)` and the result becomes `float64`: a single number decides the type when it is a higher kind (float over whole number) than the tensor, and then its own size counts.

In [11]:
scale = 0.5  # <- edit: try 2, then torch.tensor(0.5, dtype=torch.float64)
show("counts * scale", lambda: counts * scale)

counts * scale -> dtype=torch.float32 shape=(4, 24, 7) device=cpu


**Experiment 2: which views work?** The cell asks `by_day` (shape `(4, 7, 24)`, strides `(168, 1, 7)`) for a new shape. `(4, 7, 4, 6)` works: it splits the hours. Try `(4, 168)`: it fails, because it merges days and hours. Try `(28, 24)`: it fails too, because merging levels (step 168) and days (step 1) would need the level step to be 1 x 7 = 7.

In [12]:
new_shape = (4, 7, 4, 6)  # <- edit: try (4, 168), then (28, 24)
show(f"by_day.view{new_shape}", lambda: by_day.view(*new_shape))

by_day.view(4, 7, 4, 6) -> dtype=torch.int64 shape=(4, 7, 4, 6) device=cpu


**Experiment 3: make it contiguous first.** With `False`, the view fails as in step 6. Change it to `True`: `.contiguous()` copies the numbers into a new, ordered memory block, the strides become `(168, 24, 1)`, and the same `view(4, 168)` works. The cost is that copy.

In [13]:
make_contiguous = False  # <- edit: try True
src = by_day.contiguous() if make_contiguous else by_day
show("src.stride()", lambda: src.stride())
show("src.view(4, 168)", lambda: src.view(4, 168))
show("src.data_ptr() == counts.data_ptr()", lambda: src.data_ptr() == counts.data_ptr())

src.stride() -> (168, 1, 7)
src.view(4, 168) -> RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.
src.data_ptr() == counts.data_ptr() -> True


## What you should see

- `counts / 2` prints dtype `torch.float32`, while the same NumPy division gives `float64`.
- `counts.mean()` raises an error that ends in `Got: Long`; `counts.float().mean()` works.
- `by_day.stride()` prints `(168, 1, 7)` and `by_day.is_contiguous()` prints `False`.
- `by_day.view(4, 168)` raises "view size is not compatible...", and `by_day.reshape(4, 168)` works.
- After a bare `x.to(device)` line, `x.device` still prints `device(type='cpu')`.